# Training — Qwen3-0.6B, multi-source pretrain → anneal

Follows [`TRAINING_PLAN.md`](../TRAINING_PLAN.md). All heavy lifting lives in `src/`:
- `qwen3_model.py` — Qwen3-style 0.6B (RMSNorm, RoPE, GQA, SwiGLU)
- `tokenizer.py` — Qwen3 tokenizer from HF
- `data.py` — multi-source streaming (wiki/books/stackoverflow/arxiv/math) with stage mixes
- `eval.py` — per-domain val + plateau detector
- `training.py` — staged training loop

## Pipeline
1. Imports & device
2. Build model + sanity check
3. Smoke-test the multi-source loader (one batch from each source)
4. Stage 1 — pretrain (Llama 3 mix, cosine LR)
5. Stage 2 — anneal (reasoning-heavy mix, linear LR decay to 0)
6. Inspect checkpoint metadata (stage attribution, per-source token counts)

## 1 — Imports & device

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
from pathlib import Path
from importlib.metadata import version

import torch
import wandb

PROJECT_ROOT = Path.cwd().parent  # models/v2/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from qwen3_model import Qwen3Model, QWEN3_CONFIG_0_6B as CONFIG
from data import PRETRAIN_MIX, ANNEAL_MIX, build_train_loader, build_val_loaders, SOURCES
from training import default_pretrain_config, default_anneal_config, train_stage

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(123)
if device.type == "cuda":
    torch.cuda.manual_seed_all(123)
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

CKPT_DIR = PROJECT_ROOT / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

print(f"torch:    {version('torch')}")
print(f"device:   {device} ({torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu'})")
print(f"ckpt dir: {CKPT_DIR}")
print("\nmodel config:")
for k, v in CONFIG.items():
    print(f"  {k:18s} {v}")

## 2 — Build model + sanity check

In [ ]:
model = Qwen3Model(CONFIG).to(device=device, dtype=CONFIG["dtype"])
model.enable_grad_checkpointing(True)

n_total = model.num_parameters()
n_nonemb = model.num_parameters(non_embedding=True)
print(f"params (total):         {n_total:,}")
print(f"params (non-embedding): {n_nonemb:,}")

# Forward pass on random ids
with torch.no_grad():
    x = torch.randint(0, CONFIG["vocab_size"], (1, 128), device=device)
    y = model(x)
print(f"logits shape: {tuple(y.shape)}  dtype={y.dtype}")
print(f"GPU mem: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## 3 — Smoke-test the multi-source loader
Pull one batch from the pretrain mix, decode a slice, and confirm each batch element
carries a source tag. Expect first call to download/cache the first shard per source
(slow, then fast).

In [ ]:
from data import ID_TO_SOURCE
from tokenizer import decode

loader = build_train_loader(mix=PRETRAIN_MIX, context_length=CONFIG["context_length"], batch_size=4)

it = iter(loader)
x, y, src = next(it)
print(f"input shape: {tuple(x.shape)}  target shape: {tuple(y.shape)}")
print(f"source ids in this batch: {src.tolist()} -> {[ID_TO_SOURCE[i] for i in src.tolist()]}")
assert torch.equal(x[:, 1:], y[:, :-1]), "target != input shifted by 1"
print("\nsample [0], first 200 tokens:")
print(decode(x[0, :200].tolist()))

## 4 — Stage 1: pretrain
Long run with the fixed pretrain mix. Cosine LR with warmup.
Adjust `max_steps` to your time budget — the default (20 000) targets ~10B tokens at the
default effective batch size.

In [ ]:
import os, subprocess
print("kernel pid:", os.getpid())
print(subprocess.check_output(
    ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"]
).decode())

In [ ]:
pretrain_cfg = default_pretrain_config(context_length=CONFIG["context_length"])
# Override defaults here if you want — e.g.:
# pretrain_cfg.max_steps   = 5_000
# pretrain_cfg.batch_size  = 2
# pretrain_cfg.grad_accum  = 128
print(pretrain_cfg)

run = wandb.init(
    project="llm-training-v2",
    name="qwen3_0_6b_pretrain",
    config={"model": CONFIG, "stage": "pretrain", "mix": PRETRAIN_MIX.weights, **pretrain_cfg.__dict__},
)

pretrain_summary = train_stage(
    model=model, cfg=pretrain_cfg, device=device, ckpt_dir=CKPT_DIR, wandb_run=run,
)
run.finish()
print("\npretrain summary:")
for k, v in pretrain_summary.items():
    print(f"  {k}: {v}")

## 5 — Stage 2: anneal
Up-weighted reasoning mix, linear LR decay from the pretrain end-LR to 0.

In [ ]:
anneal_cfg = default_anneal_config(
    context_length=CONFIG["context_length"],
    pretrain_end_lr=pretrain_summary["final_lr"],
)
print(anneal_cfg)

run = wandb.init(
    project="llm-training-v2",
    name="qwen3_0_6b_anneal",
    config={"model": CONFIG, "stage": "anneal", "mix": ANNEAL_MIX.weights, **anneal_cfg.__dict__},
)

anneal_summary = train_stage(
    model=model, cfg=anneal_cfg, device=device, ckpt_dir=CKPT_DIR, wandb_run=run,
    starting_step=pretrain_summary["final_step"],
    starting_tokens=pretrain_summary["final_tokens"],
)
run.finish()
print("\nanneal summary:")
for k, v in anneal_summary.items():
    print(f"  {k}: {v}")

## 6 — Inspect checkpoint metadata
Every checkpoint records the stage and the exact data mix that produced it.

In [ ]:
import json

ckpt_path = CKPT_DIR / "qwen3_v2_anneal_final.pt"
payload = torch.load(ckpt_path, map_location="cpu", weights_only=False)

summary = {
    "stage":           payload["stage"],
    "stage_mix":       payload["stage_mix"],
    "step":            payload["step"],
    "tokens":          payload["tokens"],
    "docs_per_source": payload["docs_per_source"],
    "model_config":    payload["model_config_ref"],
}
print(json.dumps(summary, indent=2))

print("\nval history (last 5 evals):")
for v in payload["val_history"][-5:]:
    print(v)